# Using Generative LLM APIs in AccFin Research

This module introduces the Python client libraries for two major generative LLM providers — OpenAI and Google Gemini — covering the basic mechanics you need before using an LLM as a research tool: sending prompts, getting structured (JSON) output, holding multi-turn conversations, and passing multimodal input such as images.

**Motivation.** Generative LLMs are increasingly used in accounting research for textual-analysis tasks that used to require either dictionary/bag-of-words methods or costly manual human coding — for example, classifying disclosure tone, extracting structured facts from filings, or detecting evasive "non-answers" in earnings call Q&As. de Kok ([2025](https://doi.org/10.1287/mnsc.2023.03253)) — "ChatGPT for Textual Analysis? How to Use Generative LLMs in Accounting Research," *Management Science* 71(9), 7888–7906 — provides a framework for using these models rigorously in accounting research, covering model selection, prompt engineering, and construct validity, and illustrates it with a case study detecting non-answers in earnings conference calls. This notebook covers the API-level building blocks that a research design like that is built on top of.

**In this exercise, you'll practice:**
- Setting up API keys via `.env` for both providers.
- Sending basic prompts and reading model output.
- Requesting structured (JSON) output from a model.
- Holding a multi-turn conversation (chat).
- Passing images as multimodal input.

### 1. Using OpenAI API

For details, see offical documents at https://github.com/openai/openai-python.

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()

The best practice to handle API keys and passwords is to keep them out of your main Python scripts. The most common approach is to create a `.env` file which contains:
```
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxx
Other_Passwards=xxxxxxxxxxxxxx
```

In [ ]:
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

#### 1.1. Basic Text Prompt

In [ ]:
response = client.responses.create(
    model="gpt-5.6",
    input="Plan a one-day trip to Sydney Australia",
)
print(response.output_text)

In [ ]:
response = client.responses.create(
    model="gpt-5.6",
    instructions="You are a teaching assistant for MBA program.",
    input="Summarize the findings of Sloan (1996) 'Do Stock Prices Fully Reflect Information in Accruals and Cash Flows About Future Earnings?' in one paragraph.",
)
print(response.output_text)

In [ ]:
list_of_CEOs = ['Elon Musk', "Mark Zeckerburg"]
birthday_list = []
for ceo in list_of_CEOs:
    response = client.responses.create(
    model="gpt-5.6",
    input=f"""
    What's the birthday of {ceo}, just answer with a date in the format of "YYYYMMDD".
    """,
    )
    birthday_list.append(response.output_text)

#### 1.2. Structured JSON output

In [ ]:
class Definition(BaseModel):
    term: str
    definition: str
    confidence: float

response = client.responses.parse(
    model="gpt-5.6",
    input="Define accruals in accounting.",
    text_format=Definition,
)

# Structured Outputs guarantees a schema-conformant result - no manual JSON parsing needed
output = response.output_parsed
output

In [ ]:
# If the rest of your pipeline expects a plain dict rather than a Pydantic object:
output.model_dump()

### 2. Using Google Gemini API

- Get API Key from Google AI Studio: 

    1. Visit [Google AI Studio](https://aistudio.google.com/api-keys) and sign in with a Google account; 

    2. Create a project & get API key: click "Get API Key" (under API Access); save your API key in `.env` file.

- Install the official `google-genai` package. (The `google-generativeai` package that used to live here was fully retired by Google on November 30, 2025 — `google-genai` is its replacement.)

```bash
pip install -U "google-genai>=2.3.0"
```

  Version 2.3.0+ is required for the Interactions API used below.

In [ ]:
from google import genai

In [ ]:
client = genai.Client(api_key = os.getenv("GOOGLE_GAI_API"))

In [ ]:
# List available models:

for m in client.models.list():
    print(m.name)

#### 2.1. Basic Text Prompt

In [ ]:
interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input="Explain quantum computing in simple terms.",
)
print(interaction.output_text)

In [ ]:
# For multi-turn conversations, thread each call to the previous one via previous_interaction_id
# (the Interactions API manages conversation history server-side)

interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input="What's the capital of France?",
)
print(interaction.output_text)

In [ ]:
interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input="What's its population?",
    previous_interaction_id=interaction.id,
)
print(interaction.output_text)

#### 2.2 Multimodal Input (e.g., Images)

In [ ]:
import base64

with open("path_to_image.jpg", "rb") as f:
    image_bytes = f.read()

interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input=[
        {"type": "text", "text": "Describe what you see."},
        {"type": "image", "mime_type": "image/jpeg", "data": base64.b64encode(image_bytes).decode("utf-8")},
    ],
)
print(interaction.output_text)

Resources:
* [Google Gen AI SDK for Python](https://github.com/googleapis/python-genai)
* [Gemini API Reference](https://ai.google.dev/api)
* [Interactions API guide](https://ai.google.dev/gemini-api/docs/interactions)
* [AI Studio](https://aistudio.google.com/)